In [1]:
from torchvision.datasets import OxfordIIITPet

In [3]:
train_dataset = OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="segmentation",
    download=True
)

test_dataset = OxfordIIITPet(
    root="./data",
    split="test",
    target_types="segmentation",
    download=True
)

In [11]:
from torch.utils.data import Subset
import random

target_size = (500, 375)
keep_ratio = 0.20

matching_indices = []

for idx in range(len(train_dataset)):
    image, mask = train_dataset[idx]

    image_size = image.shape[-2:] if hasattr(image, "shape") else image.size

    if image_size == target_size:
        matching_indices.append(idx)

num_keep = int(len(matching_indices) * keep_ratio)

random.seed(42)
selected_indices = random.sample(matching_indices, num_keep)

train_subset = Subset(train_dataset, selected_indices)

print("Original matching images:", len(matching_indices))
print("Selected training images:", len(train_subset))

print(train_subset[0])

Original matching images: 770
Selected training images: 154
(<PIL.Image.Image image mode=RGB size=500x375 at 0x11961C9A0>, <PIL.PngImagePlugin.PngImageFile image mode=L size=500x375 at 0x119789970>)


In [12]:
print(train_subset[0])

(<PIL.Image.Image image mode=RGB size=500x375 at 0x1196151F0>, <PIL.PngImagePlugin.PngImageFile image mode=L size=500x375 at 0x119715670>)


In [ ]:
## loss using IOU or Dice // IOU  = Intersection / Union and dice = 2 * Intersection / (Size A + Size B)
## image -- > conv2D -> relu --relu--> conv2D -- forCcat , maxPool (2x2) 

## 393x500 --> 196x250 --> 98x125

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
## conv2D->relu->conv2D-->relu
class DoubleConv(nn.Module):
    def __init__(self, in_channels,out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

In [17]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.down1 = DoubleConv(3, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)

        self.bottleneck = DoubleConv(512, 1024)

    def forward(self, x):
        skip1 = self.down1(x)
        x = self.pool(skip1)

        skip2 = self.down2(x)
        x = self.pool(skip2)

        skip3 = self.down3(x)
        x = self.pool(skip3)

        skip4 = self.down4(x)
        x = self.pool(skip4)

        x = self.bottleneck(x)

        return x, skip1, skip2, skip3, skip4

In [18]:
encoder = Encoder()

x = torch.randn(1, 3, 512, 384)

latent, skip1, skip2, skip3, skip4 = encoder(x)

print("skip1:", skip1.shape)
print("skip2:", skip2.shape)
print("skip3:", skip3.shape)
print("skip4:", skip4.shape)
print("latent:", latent.shape)

skip1: torch.Size([1, 64, 512, 384])
skip2: torch.Size([1, 128, 256, 192])
skip3: torch.Size([1, 256, 128, 96])
skip4: torch.Size([1, 512, 64, 48])
latent: torch.Size([1, 1024, 32, 24])
